# ⚡ Opus2Short AI Studio — 1-Clique no Google Colab (Grátis + Google Drive + Domínio Fixo)

Este notebook sobe o **Opus2Short AI Studio** (clone full-stack do OpusClip + 2Short.ai) gratuitamente no Google Colab:
- Salva projetos, clipes, Brand Kits e API Keys permanentemente no seu **Google Drive** (`MyDrive/Opus2ShortStudio`).
- Importa automaticamente `client_secret*.json` da pasta **`MyDrive/KEYS`**.
- Suporta **Domínio Fixo Grátis do Ngrok** (`.ngrok-free.app` / `.ngrok-free.dev`) ou fallback automático para `trycloudflare.com`.

In [ ]:
# @title 1. Conectar Google Drive e importar credenciais de MyDrive/KEYS
import os, glob, shutil
from google.colab import drive

drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/Opus2ShortStudio'
KEYS_DIR = '/content/drive/MyDrive/KEYS'

os.makedirs(f'{DRIVE_BASE}/data', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/storage/uploads', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/storage/projects', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/storage/exports', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/storage/brandkits', exist_ok=True)

found_cookies = glob.glob(f"{KEYS_DIR}/*cookie*.txt") + glob.glob(f"{KEYS_DIR}/cookies.txt")
if found_cookies:
    shutil.copy2(found_cookies[0], f"{DRIVE_BASE}/data/cookies.txt")
    print(f"🍪 Cookies do YouTube importados: {os.path.basename(found_cookies[0])}")

found_secrets = glob.glob(f'{KEYS_DIR}/client_secret*.json') + glob.glob(f'{KEYS_DIR}/*client_secret*.json')
if found_secrets:
    src = found_secrets[0]
    dst = f'{DRIVE_BASE}/data/client_secret.json'
    shutil.copy2(src, dst)
    print(f'🔑 Credencial Google Cloud importada automaticamente: {os.path.basename(src)}')
else:
    print(f'ℹ️ Nenhum client_secret*.json encontrado em {KEYS_DIR} (você também pode enviar pela aba PlanSocial).')

print('✅ Google Drive pronto em:', DRIVE_BASE)

In [ ]:
# @title 2. Instalar dependências e iniciar o Studio (Link Fixo Ngrok opcional ou Cloudflare)
NGROK_AUTHTOKEN = "" # @param {type:"string"}
NGROK_DOMAIN = "" # @param {type:"string"} (ex: meu-dominio.ngrok-free.app — NÃO use .internal)

import os, subprocess, time, re

REPO_DIR = '/content/rea-opusclip-and-2short'
DRIVE_BASE = '/content/drive/MyDrive/Opus2ShortStudio'

print('[1/5] Instalando Node.js 22, FFmpeg e fontes...')
!apt-get update -qq && apt-get install -y -qq ffmpeg libass-dev fonts-dejavu-core fonts-liberation curl >/dev/null
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1 && apt-get install -y -qq nodejs >/dev/null
!node -v && ffmpeg -version | head -n 1

print('[2/5] Instalando yt-dlp, faster-whisper e OpenCV...')
!pip install -q yt-dlp faster-whisper opencv-python-headless edge-tts

print('[3/5] Baixando/atualizando repositório Opus2Short Studio...')
if os.path.exists(f'{REPO_DIR}/.git'):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/LANOSGITUFC/rea-opusclip-and-2short.git {REPO_DIR}

!rm -rf {REPO_DIR}/app/data {REPO_DIR}/app/storage
!ln -s {DRIVE_BASE}/data {REPO_DIR}/app/data
!ln -s {DRIVE_BASE}/storage {REPO_DIR}/app/storage

print('[4/5] Compilando Frontend React e instalando Backend...')
!npm --prefix {REPO_DIR}/app/web install --no-audit --no-fund >/dev/null
!npm --prefix {REPO_DIR}/app/web run build
!npm --prefix {REPO_DIR}/app/server install --omit=dev --no-audit --no-fund >/dev/null

print('[5/5] Iniciando servidor na porta 3000 e abrindo túnel HTTPS...')
!pkill -f 'app/server/index.js' 2>/dev/null || true
!pkill -f 'cloudflared tunnel' 2>/dev/null || true
!pkill -f 'ngrok http' 2>/dev/null || true

server_proc = subprocess.Popen(
    ['node', f'{REPO_DIR}/app/server/index.js'],
    cwd=f'{REPO_DIR}/app/server',
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT
)
time.sleep(2)

public_url = None
if NGROK_AUTHTOKEN.strip():
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    dom = NGROK_DOMAIN.strip().replace('https://', '').replace('http://', '').strip('/')
    if dom and not dom.endswith('.internal'):
        tunnel = ngrok.connect(3000, domain=dom)
    else:
        if dom.endswith('.internal'):
            print('⚠️ Domínios .internal são privados. Usando domínio público do Ngrok...')
        tunnel = ngrok.connect(3000)
    public_url = tunnel.public_url
else:
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3000'],
        stdout=open('/content/tunnel.log', 'w'),
        stderr=subprocess.STDOUT
    )
    for _ in range(20):
        time.sleep(1)
        with open('/content/tunnel.log', 'r', errors='ignore') as f:
            m = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', f.read())
            if m:
                public_url = m.group(0)
                break

print('\n' + '='*60)
if public_url:
    print('🚀 OPUS2SHORT AI STUDIO NO AR! ABRA NO NOTEBOOK OU CELULAR:')
    print(f'👉 {public_url}')
    print(f'🔑 Redirect URI p/ Google Cloud Console:')
    print(f'   {public_url}/api/oauth/youtube/callback')
else:
    print('⚠️ Não foi possível ler a URL. Veja /content/tunnel.log')
print('='*60)
print('Mantenha esta aba aberta enquanto estiver usando o Studio.')